# EEG Depression Detection — full pipeline in one notebook

This runs the **entire** pipeline (the same stages as `scripts/run_pipeline.py`):

1. Preprocess raw `.edf` → cleaned, epoched EEG  (`build_epoch_dataset`)
2. Epochs → time-frequency images  (`build_tfd_dataset`)
3. Subject-wise k-fold training + evaluation  (`train_one_fold` / `evaluate_fold`)
4. Aggregate + write `results/`  (`aggregate_and_report`)

Stages 1 and 2 are **cached** on disk (`data/processed/`), so re-runs skip straight to training.

**How to use:** pick a kernel that has the project deps installed, then Run All.
Start with `QUICK_TEST = True` (a tiny, meaningless-but-fast run to prove the plumbing works),
then set it to `False` for the real run.

## 0. (Optional) install dependencies

Only needed once, in a fresh environment. The versions pinned in `requirements.txt` may not resolve
(they look like future/placeholder versions), so install unpinned. **Restart the kernel afterwards.**

In [ ]:
# %pip install mne PyWavelets numpy scipy scikit-learn scikit-image pandas matplotlib PyYAML torch torchvision tqdm

## 1. Locate repo root + imports

The `src` code builds paths relative to the working directory (`data/raw`, `configs/default.yaml`, ...),
so we `chdir` to the folder that contains `src/` and `configs/` no matter where the notebook is launched from.

In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
if not ((ROOT / "src").is_dir() and (ROOT / "configs").is_dir()):
    for cand in list(pathlib.Path.cwd().parents):
        if (cand / "src").is_dir() and (cand / "configs").is_dir():
            ROOT = cand
            break
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root / working dir:", os.getcwd())
assert (ROOT / "src").is_dir(), "could not find the project's src/ folder"

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import load_config
from src.preprocessing import discover_recordings, build_epoch_dataset
from src.tfd_transform import build_tfd_dataset
from src.dataset import make_subject_kfold_splits
from src.train import get_device, set_seed, train_one_fold
from src.evaluate import evaluate_fold, aggregate_and_report

print("imports OK")

## 2. Load config + notebook knobs

Everything tunable lives in `configs/default.yaml`. Here we only override things that matter for
running inside a notebook on Windows, plus the `QUICK_TEST` switch.

In [ ]:
cfg = load_config("configs/default.yaml")

# Windows + Jupyter: multiprocessing DataLoader workers are unreliable here -> use the main process.
cfg.training.num_workers = 0

# QUICK_TEST=True  -> 2 folds, 1 epoch each: proves the whole path runs, numbers are NOT meaningful.
# QUICK_TEST=False -> the real run from configs/default.yaml (5 folds x 20 epochs). Slow on CPU (hours).
QUICK_TEST = True

if QUICK_TEST:
    cfg.training.k_folds = 2
    cfg.training.epochs = 1
    cfg.training.early_stopping_patience = 1

set_seed(cfg.training.seed)
device = get_device()
print(f"device={device}  quick_test={QUICK_TEST}  k_folds={cfg.training.k_folds}  epochs={cfg.training.epochs}")
print(f"transform={cfg.tfd.transform}  backbone={cfg.model.backbone}")

## 3. Stage 1 — preprocessing (load / filter / ICA / epoch)

Reads `data/processed/epochs_metadata.csv` if present (it is), so this returns instantly.
Pass `force=True` to re-run from the raw `.edf` files (slow — ICA on 58 recordings).

In [ ]:
recordings = discover_recordings(cfg.data.raw_dir, cfg.data.condition,
                                 cfg.data.mdd_prefix, cfg.data.hc_prefix)
print(f"{len(recordings)} '{cfg.data.condition}' recordings discovered in {cfg.data.raw_dir}")

epoch_metadata = build_epoch_dataset(cfg, force=False)
print(f"{len(epoch_metadata)} epochs across {epoch_metadata['subject_id'].nunique()} subjects")
epoch_metadata.groupby('group')['subject_id'].nunique()

## 4. Stage 2 — time-frequency images

Reads `data/processed/tfd_metadata_cwt.csv` if present. `force=True` regenerates every image
(needed if you change any `tfd:` setting or switch `transform` to `spwvd`).

In [ ]:
tfd_metadata = build_tfd_dataset(cfg, epoch_metadata, force=False)
print(f"{len(tfd_metadata)} TFD images ({cfg.tfd.transform})")
tfd_metadata.head()

In [ ]:
# Peek at one HC image and one MDD image
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, lbl, name in [(axes[0], 0, "HC"), (axes[1], 1, "MDD")]:
    row = tfd_metadata[tfd_metadata.label == lbl].iloc[0]
    ax.imshow(np.load(row["tfd_path"]), origin="lower", aspect="auto", cmap="viridis")
    ax.set_title(f"{name}: {row['subject_id']} epoch {row['epoch_index']}")
fig.tight_layout()

## 5. Stage 3 — subject-wise k-fold training + evaluation

This is the slow part. The first run also downloads the pretrained backbone weights (~100 MB for ResNet50).
Each fold prints its per-epoch train/val loss, then its held-out test metrics.

In [ ]:
fold_metrics = []
splits = make_subject_kfold_splits(tfd_metadata, cfg.training.k_folds, cfg.training.seed)
for fold_idx, (train_meta, test_meta) in enumerate(splits):
    print(f"\n--- Fold {fold_idx}: {train_meta['subject_id'].nunique()} train / "
          f"{test_meta['subject_id'].nunique()} test subjects ---")
    model = train_one_fold(cfg, train_meta, fold_idx, device)
    m = evaluate_fold(model, test_meta, cfg, device)
    print(f"fold {fold_idx}: acc={m['accuracy']:.4f} prec={m['precision']:.4f} "
          f"rec={m['recall']:.4f} f1={m['f1']:.4f}")
    fold_metrics.append(m)

## 6. Aggregate + write `results/`

Writes `results/metrics.json`, `results/per_fold_metrics.csv`, and confusion-matrix PNGs.

In [ ]:
summary = aggregate_and_report(fold_metrics, cfg)
pd.DataFrame(fold_metrics)[["accuracy", "precision", "recall", "f1", "n_test_epochs", "n_test_subjects"]]

In [ ]:
print(json.dumps(summary["summary_mean_std"], indent=2))
for p in sorted(glob.glob("results/confusion_matrix_*.png")):
    print(p)
    display(Image(p))

## Notes

- **Full run is slow on CPU.** ResNet50 fine-tuning on ~8k images x 20 epochs x 5 folds is hours on CPU,
  minutes-to-hours on a CUDA GPU (PyTorch uses it automatically if present).
- **Change settings** by editing `cfg` after the load cell (e.g. `cfg.tfd.transform = "spwvd"`,
  `cfg.model.backbone = "vgg16"`) — then re-run stage 2 with `build_tfd_dataset(cfg, epoch_metadata, force=True)`
  if you touched anything under `tfd:`.
- **Regenerate caches** from scratch: `build_epoch_dataset(cfg, force=True)` then
  `build_tfd_dataset(cfg, epoch_metadata, force=True)`.
- The raw `.edf` files must be in `data/raw/` (they are).